In [0]:
%sql
-- 1. Limpiar la estructura inicial (está vacía)
DROP SCHEMA IF EXISTS workspace.bakehouse CASCADE;

-- 2. Un schema por capa
CREATE SCHEMA IF NOT EXISTS workspace.raw    COMMENT 'Zona de aterrizaje de archivos de sistemas origen';
CREATE SCHEMA IF NOT EXISTS workspace.bronze COMMENT 'Datos tal como llegan + campos de auditoría';
CREATE SCHEMA IF NOT EXISTS workspace.silver COMMENT 'Datos limpios, tipados y deduplicados';
CREATE SCHEMA IF NOT EXISTS workspace.gold   COMMENT 'Modelo estrella para consumo y BI';
CREATE SCHEMA IF NOT EXISTS workspace.ops    COMMENT 'Auditoría, logs y control de ejecuciones';

-- 3. Volume de aterrizaje
CREATE VOLUME IF NOT EXISTS workspace.raw.landing COMMENT 'Archivos entrantes (JSON) del sistema de ventas';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.ops.pipeline_runs (
  run_id        STRING    COMMENT 'Id único de la ejecución de esta tarea',
  job_run_id    STRING    COMMENT 'Id del Job o de ADF que la lanzó',
  pipeline      STRING    COMMENT 'Nombre del pipeline',
  task          STRING    COMMENT 'Notebook o tarea',
  layer         STRING    COMMENT 'raw | bronze | silver | gold',
  target_table  STRING,
  status        STRING    COMMENT 'RUNNING | SUCCESS | FAILED',
  rows_read     BIGINT,
  rows_written  BIGINT,
  rows_rejected BIGINT,
  error_message STRING,
  started_at    TIMESTAMP,
  finished_at   TIMESTAMP,
  executed_by   STRING
) COMMENT 'Log de ejecuciones del pipeline';

CREATE TABLE IF NOT EXISTS workspace.ops.data_quality_issues (
  run_id         STRING    COMMENT 'Ejecución que detectó el problema',
  layer          STRING,
  target_table   STRING,
  rule_name      STRING    COMMENT 'Regla de calidad incumplida',
  record_key     STRING    COMMENT 'Clave del registro rechazado',
  record_payload STRING    COMMENT 'Registro completo en JSON',
  detected_at    TIMESTAMP
) COMMENT 'Registros rechazados por reglas de calidad';

In [0]:
%sql
-- Volume para los checkpoints de streaming (separado de los datos)
CREATE VOLUME IF NOT EXISTS workspace.ops.checkpoints
  COMMENT 'Estado interno de Auto Loader y streaming';

-- Vista con el estado final de cada ejecución
CREATE OR REPLACE VIEW workspace.ops.v_pipeline_runs
COMMENT 'Última foto de cada ejecución (el log es append-only)'
AS
SELECT * EXCEPT (rn) FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY run_id ORDER BY COALESCE(finished_at, started_at) DESC) AS rn
  FROM workspace.ops.pipeline_runs
) WHERE rn = 1;

In [0]:
%sql
-- ── Tabla silver de ventas: el contrato de datos ─────────────────────────
-- Se define a mano (no se infiere): tipos, nombres en snake_case y comentarios.
-- PRIMARY KEY: informativa (Databricks no la hace cumplir), pero documenta la clave
--   y la usan el optimizador y las herramientas de BI.
-- CLUSTER BY: liquid clustering por fecha (organiza los archivos para filtrar rápido por fecha).
CREATE TABLE IF NOT EXISTS workspace.silver.transactions (
  transaction_id      BIGINT        NOT NULL COMMENT 'Id de la venta (clave de negocio)',
  customer_id         BIGINT        NOT NULL COMMENT 'Cliente',
  franchise_id        BIGINT                 COMMENT 'Franquicia (tienda)',
  transaction_ts      TIMESTAMP              COMMENT 'Fecha y hora de la venta (UTC)',
  transaction_date    DATE                   COMMENT 'Fecha de la venta',
  product             STRING                 COMMENT 'Producto',
  quantity            INT                    COMMENT 'Unidades vendidas',
  unit_price          DECIMAL(10,2)          COMMENT 'Precio unitario',
  total_price         DECIMAL(12,2)          COMMENT 'Importe total',
  payment_method      STRING                 COMMENT 'Medio de pago',
  discount_pct        DECIMAL(5,2)           COMMENT 'Descuento (solo desde que el origen lo envía)',
  created_at          TIMESTAMP              COMMENT 'Alta en el sistema origen',
  updated_at          TIMESTAMP              COMMENT 'Última modificación en el origen (NULL si nunca)',
  _change_ts          TIMESTAMP              COMMENT 'COALESCE(updated_at, created_at): versión del registro',
  _source_file        STRING                 COMMENT 'Archivo de origen de esta versión',
  _bronze_ingested_at TIMESTAMP              COMMENT 'Cuándo llegó a bronze',
  _run_id             STRING                 COMMENT 'Ejecución que escribió esta versión',
  _silver_updated_at  TIMESTAMP              COMMENT 'Cuándo se escribió en silver',
  CONSTRAINT pk_silver_transactions PRIMARY KEY (transaction_id)
)
CLUSTER BY (transaction_date)
COMMENT 'Ventas limpias, tipadas y deduplicadas: una fila por transaction_id (última versión)';

-- ── Regla de calidad dentro de la tabla (CHECK constraint) ───────────────
-- A diferencia de la PRIMARY KEY, Delta SÍ la hace cumplir: si alguien intenta
-- escribir quantity <= 0, la escritura entera falla. Es la última línea de defensa.
-- DROP + ADD para que la celda se pueda re-ejecutar sin error.
ALTER TABLE workspace.silver.transactions DROP CONSTRAINT IF EXISTS quantity_positiva;
ALTER TABLE workspace.silver.transactions ADD CONSTRAINT quantity_positiva CHECK (quantity > 0);

-- ── Tabla de control de watermarks (el patrón que usabas en Fabric) ──────
CREATE TABLE IF NOT EXISTS workspace.ops.watermarks (
  source_table  STRING    COMMENT 'Tabla leída (origen de la carga incremental)',
  target_table  STRING    COMMENT 'Tabla destino',
  watermark_ts  TIMESTAMP COMMENT 'Último _ingested_at de bronze procesado con éxito',
  run_id        STRING    COMMENT 'Ejecución que movió el watermark',
  updated_at    TIMESTAMP COMMENT 'Cuándo se actualizó'
) COMMENT 'Watermarks de las cargas incrementales entre capas';